In [149]:
# Peloton: Find out why subscribers of a fitness app cancel, and give leadership an evidence-backed plan to keep them.

import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [150]:
# Load and clean the subscriptions and usage datasets. Check for repeated records, inconsistent labels and missing values, and explain how you handled each.

In [151]:
subscriptions = pd.read_csv(r'/Users/karanthapa/Desktop/customer-churn-analysis/datasets/subscriptions.csv')
usage = pd.read_csv(r'/Users/karanthapa/Desktop/customer-churn-analysis/datasets/usage.csv')

In [152]:
# creating the copy of each dataset 
sub_df = subscriptions.copy()
use_df = usage.copy()

In [153]:
sub_df.head(10)

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaN,NaN
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaN,NaN
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
4,5,2025-06-24,annual,plus,14.99,paid_social,android,55+,no,NaN,NaN
5,6,2024-08-11,monthly,plus,19.99,organic,web,55+,yes,2025-03-01,not_using_enough
6,7,2024-11-05,monthly,plus,19.99,organic,android,25-34,yes,2024-12-28,other
7,8,2025-10-20,annual,plus,14.99,organic,web,45-54,no,NaN,NaN
8,9,2024-08-29,annual,basic,9.74,promo_offer,ios,25-34,yes,2025-07-27,too_expensive
9,10,2025-01-26,annual,basic,9.74,organic,ios,35-44,no,NaN,NaN


In [154]:
use_df.head(10)

,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets
0,1,2024-09,5,140.0,0,0
1,1,2024-10,4,116.0,0,0
2,1,2024-11,7,208.0,0,0
3,1,2024-12,5,109.0,0,0
4,1,2025-01,1,41.0,0,0
5,1,2025-02,6,119.0,0,0
6,1,2025-03,5,167.0,0,3
7,1,2025-04,3,78.0,0,2
8,1,2025-05,7,245.0,0,0
9,1,2025-06,3,111.0,0,1


In [155]:
sub_df.shape, use_df.shape

((4200, 11), (30685, 6))

# Checking the null values 

In [156]:
sub_df.isnull().sum()

customer_id               0
signup_date               0
plan_type                 0
plan_tier                 0
monthly_price             0
acquisition_channel       0
primary_device            0
age_group                 0
churned                   0
churn_date             2432
cancel_reason          2814
dtype: int64

In [157]:
use_df.isnull().sum()

customer_id             0
month                   0
workouts_completed      0
minutes_active        506
classes_booked          0
support_tickets         0
dtype: int64

In [158]:
sub_df['cancel_reason'] = sub_df['cancel_reason'].where(
    sub_df['churned'] == 'no', sub_df['cancel_reason'].fillna('unknown')
).fillna('not_churned')

In [159]:
use_df['minutes_missing'] = use_df['minutes_active'].isna().astype(int)
use_df['minutes_active'] = use_df['minutes_active'].fillna(use_df['workouts_completed'] * 30)


# checking the duplicates 

In [160]:
sub_df.duplicated().sum()

np.int64(0)

In [161]:
use_df.duplicated().sum()

np.int64(160)

# deleting the duplicates 

In [162]:
sub_df.drop_duplicates(inplace=True)
use_df.drop_duplicates(inplace=True)

In [163]:
use_df.duplicated().sum()

np.int64(0)

In [164]:
sub_df.duplicated().sum()

np.int64(0)

# checking the datatype of sub_df

In [165]:
sub_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          4200 non-null   int64  
 1   signup_date          4200 non-null   object 
 2   plan_type            4200 non-null   object 
 3   plan_tier            4200 non-null   object 
 4   monthly_price        4200 non-null   float64
 5   acquisition_channel  4200 non-null   object 
 6   primary_device       4200 non-null   object 
 7   age_group            4200 non-null   object 
 8   churned              4200 non-null   object 
 9   churn_date           1768 non-null   object 
 10  cancel_reason        4200 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 361.1+ KB


# changing signup_date and churn_date from object Dtype to datetime Dtype

In [166]:
sub_df['signup_date'] = pd.to_datetime(sub_df['signup_date'])

sub_df['churn_date'] = pd.to_datetime(sub_df['churn_date'])

# changing age_group from oject Dtype to Categorical Dtype

In [167]:
order = sub_df['age_group'].unique()
sub_df['age_group'] = pd.Categorical(sub_df['age_group'], categories=order, ordered=True)

# checking the Dtype for use_df

In [168]:
use_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30525 entries, 0 to 30684
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         30525 non-null  int64  
 1   month               30525 non-null  object 
 2   workouts_completed  30525 non-null  int64  
 3   minutes_active      30525 non-null  float64
 4   classes_booked      30525 non-null  int64  
 5   support_tickets     30525 non-null  int64  
 6   minutes_missing     30525 non-null  int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 1.9+ MB


# changing month from Object Dtype to Datetime Dtyep

In [169]:
use_df['month'] = pd.to_datetime(use_df['month'], format='%Y-%m')

# making all the values of plan_tier to lower case

In [170]:
sub_df['plan_tier'] = sub_df['plan_tier'].str.lower()

# customer who churned 

In [171]:
sub_df[sub_df['churned'] == 'yes']

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
5,6,2024-08-11,monthly,plus,19.99,organic,web,55+,yes,2025-03-01,not_using_enough
6,7,2024-11-05,monthly,plus,19.99,organic,android,25-34,yes,2024-12-28,other
8,9,2024-08-29,annual,basic,9.74,promo_offer,ios,25-34,yes,2025-07-27,too_expensive
...,...,...,...,...,...,...,...,...,...,...,...
4186,4187,2025-08-19,annual,basic,9.74,promo_offer,android,25-34,yes,2025-12-14,too_expensive
4187,4188,2024-08-28,monthly,basic,12.99,partner,android,35-44,yes,2024-08-28,technical_issues
4189,4190,2025-02-07,monthly,plus,19.99,organic,android,45-54,yes,2025-02-21,too_expensive
4190,4191,2024-09-07,monthly,plus,19.99,organic,ios,25-34,yes,2024-11-22,not_using_enough


In [172]:
sub_df[sub_df['churned'] == 'yes']

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
5,6,2024-08-11,monthly,plus,19.99,organic,web,55+,yes,2025-03-01,not_using_enough
6,7,2024-11-05,monthly,plus,19.99,organic,android,25-34,yes,2024-12-28,other
8,9,2024-08-29,annual,basic,9.74,promo_offer,ios,25-34,yes,2025-07-27,too_expensive
...,...,...,...,...,...,...,...,...,...,...,...
4186,4187,2025-08-19,annual,basic,9.74,promo_offer,android,25-34,yes,2025-12-14,too_expensive
4187,4188,2024-08-28,monthly,basic,12.99,partner,android,35-44,yes,2024-08-28,technical_issues
4189,4190,2025-02-07,monthly,plus,19.99,organic,android,45-54,yes,2025-02-21,too_expensive
4190,4191,2024-09-07,monthly,plus,19.99,organic,ios,25-34,yes,2024-11-22,not_using_enough


In [173]:
sub_df['tenure_in_days'] = (sub_df['churn_date'].fillna(sub_df['churn_date'].max()) - sub_df['signup_date']).dt.days

# Explore churn rates across plan type, plan tier, acquisition channel, device and age group, and across tenure.

## Hint : Members who joined recently haven't had much time to cancel, so their overall churn looks low. A monthly churn rate compares groups fairly: cancellations in a month divided by the members who were active that month

In [174]:
sub_df.head(3)

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason,tenure_in_days
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues,291
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaT,not_churned,139
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaT,not_churned,341


In [175]:
months = pd.period_range(sub_df['signup_date'].min(), sub_df['churn_date'].max(), freq='M')
rows = []
for m in months:
    active = sub_df[(sub_df['signup_date'] <= m.end_time) &
                    (sub_df['churn_date'].isna() | (sub_df['churn_date'] >= m.start_time))]
    cancelled = active[active['churn_date'].between(m.start_time, m.end_time)]
    rows.append({'month': m, 'active': len(active), 'cancelled': len(cancelled)})

monthly = pd.DataFrame(rows)
monthly['churn_rate'] = monthly['cancelled'] / monthly['active']
monthly

,month,active,cancelled,churn_rate
0,2024-07,172,8,0.046512
1,2024-08,359,14,0.038997
2,2024-09,572,38,0.066434
3,2024-10,785,36,0.045860
4,2024-11,999,52,0.052052
5,2024-12,1232,54,0.043831
6,2025-01,1427,77,0.053959
7,2025-02,1601,81,0.050593
8,2025-03,1804,103,0.057095
9,2025-04,1965,98,0.049873


In [179]:
df = sub_df.join(monthly)
df.head(3)

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason,tenure_in_days,month,active,cancelled,churn_rate
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues,291,2024-07,172.0,8.0,0.046512
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaT,not_churned,139,2024-08,359.0,14.0,0.038997
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaT,not_churned,341,2024-09,572.0,38.0,0.066434


In [177]:
sub_df['signup_date'] = pd.to_datetime(sub_df['signup_date'])
sub_df['churn_date'] = pd.to_datetime(sub_df['churn_date'])

months = pd.period_range(sub_df['signup_date'].min(), sub_df['churn_date'].max(), freq='M')

def monthly_churn(df):
    active_total = 0
    cancelled_total = 0
    for m in months:
        active = df[(df['signup_date'] <= m.end_time) &
                    (df['churn_date'].isna() | (df['churn_date'] >= m.start_time))]
        cancelled = active[active['churn_date'].between(m.start_time, m.end_time)]
        active_total += len(active)
        cancelled_total += len(cancelled)
    return round(100 * cancelled_total / active_total, 1)

sub_df.groupby('plan_type').apply(monthly_churn)

/var/folders/z3/0l9hbnmx27qg6j4p8vp3gzzr0000gn/T/ipykernel_6552/337905836.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_df.groupby('plan_type').apply(monthly_churn)


plan_type
annual     3.0
monthly    7.5
dtype: float64

In [180]:
df.head(3)

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason,tenure_in_days,month,active,cancelled,churn_rate
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues,291,2024-07,172.0,8.0,0.046512
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaT,not_churned,139,2024-08,359.0,14.0,0.038997
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaT,not_churned,341,2024-09,572.0,38.0,0.066434


In [188]:
df1 = df.copy()

In [195]:
df1

,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason,tenure_in_days,month,active,cancelled,churn_rate
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues,291,2024-07,172.0,8.0,4.651163
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaT,not_churned,139,2024-08,359.0,14.0,3.899721
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaT,not_churned,341,2024-09,572.0,38.0,6.643357
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough,5,2024-10,785.0,36.0,4.585987
4,5,2025-06-24,annual,plus,14.99,paid_social,android,55+,no,NaT,not_churned,190,2024-11,999.0,52.0,5.205205
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,4196,2025-03-01,annual,basic,9.74,partner,ios,25-34,no,NaT,not_churned,305,NaT,NaN,NaN,NaN
4196,4197,2025-03-09,annual,basic,9.74,organic,ios,25-34,no,NaT,not_churned,297,NaT,NaN,NaN,NaN
4197,4198,2024-12-01,annual,plus,14.99,promo_offer,android,35-44,no,NaT,not_churned,395,NaT,NaN,NaN,NaN
4198,4199,2024-09-06,monthly,plus,19.99,promo_offer,ios,18-24,no,NaT,not_churned,481,NaT,NaN,NaN,NaN
